# Peptide RT — RDKit-only baseline + ChemBERTa-Concat ablation

Peptide-domain twin of `chemberta_concat_baseline_lipid.ipynb`. Same three new model conditions, same evaluation protocol, but trained on the ~250k ProteomeTools peptide RT split that the paper uses for multi-task pre-training.

1. **RDKit-only baseline** — Ridge, RandomForest, GradientBoosting on the seven RDKit descriptors. Floor for how much peptide RT signal lives in the rule-based descriptors alone.
2. **ChemBERTa-Concat** — `[CLS] (768) ‖ scaled RDKit (7) → Linear → RT`, single-task MSE. Same hyperparameters as the paper's multi-task model.
3. **ChemBERTa-Concat (frozen)** — Same architecture, but ChemBERTa weights frozen and only the linear head trains.

The peptide CSVs (`peptide250_train_with_rdkit.csv`, `peptide250_test_with_rdkit.csv`) are not in the repo — upload them in the Colab cell below, same as the existing peptide notebooks.

Manuscript reference numbers for this split: ChemBERTa-RT R²=0.743, MAE=295.6 — ChemBERTa+RDKit R²=0.757, MAE=258.3.

In [ ]:
!pip install torch transformers scikit-learn matplotlib joblib

In [ ]:
# Upload peptide250_train_with_rdkit.csv and peptide250_test_with_rdkit.csv
n = 1
while n < 2:
    from google.colab import files
    uploaded = files.upload()
    n += 1

In [ ]:
import pandas as pd
import numpy as np

train_df = pd.read_csv('/content/peptide250_train_with_rdkit.csv')
test_df  = pd.read_csv('/content/peptide250_test_with_rdkit.csv')
print('train:', len(train_df), '  test:', len(test_df))

DESC_COLS = ['mol_weight', 'polar_surface_area', 'h_bond_donors',
             'h_bond_acceptors', 'rotatable_bonds', 'aromatic_rings', 'heavy_atoms']

## 1. RDKit-only baseline

No ChemBERTa, no SMILES tokens — just the seven descriptors → RT.

In [ ]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error

X_train = train_df[DESC_COLS].values.astype(float)
X_test  = test_df[DESC_COLS].values.astype(float)
y_train = train_df['rt'].values.astype(float)
y_test  = test_df['rt'].values.astype(float)

x_scaler = MinMaxScaler().fit(X_train)
X_train_s = x_scaler.transform(X_train)
X_test_s  = x_scaler.transform(X_test)

y_scaler = MinMaxScaler()
y_train_s = y_scaler.fit_transform(y_train.reshape(-1, 1)).ravel()
y_test_s  = y_scaler.transform(y_test.reshape(-1, 1)).ravel()

BASELINE_SEEDS = [0, 1, 2, 3, 4]
rows = []

for seed in BASELINE_SEEDS:
    candidates = [
        ('Ridge',            Ridge(alpha=1.0)),
        ('RandomForest',     RandomForestRegressor(n_estimators=200, random_state=seed, n_jobs=-1)),
        ('GradientBoosting', GradientBoostingRegressor(n_estimators=200, random_state=seed)),
    ]
    for name, mdl in candidates:
        mdl.fit(X_train_s, y_train_s)
        for split_name, X_s, y_orig in [('Train', X_train_s, y_train), ('Test', X_test_s, y_test)]:
            y_pred_s = mdl.predict(X_s)
            y_pred   = y_scaler.inverse_transform(y_pred_s.reshape(-1, 1)).ravel()
            rows.append({'model': f'RDKit-only ({name})', 'seed': seed, 'split': split_name,
                         'metric': 'R2',  'value': r2_score(y_orig, y_pred)})
            rows.append({'model': f'RDKit-only ({name})', 'seed': seed, 'split': split_name,
                         'metric': 'MAE', 'value': mean_absolute_error(y_orig, y_pred)})

baseline_df = pd.DataFrame(rows)
baseline_df.to_csv('peptide_rdkit_only_baseline_metrics.csv', index=False)
print(baseline_df.pivot_table(index='model', columns=['split', 'metric'], values='value',
                              aggfunc=['median', 'std']).round(4))

## 2. ChemBERTa-Concat (fine-tuned and frozen variants)

[CLS] embedding (768) ‖ scaled RDKit descriptors (7) → Linear → RT. Single-task MSE on RT. Same hyperparameters as the paper's other ChemBERTa runs (AdamW lr=2.5e-5, batch=16, 15 epochs).

- **ChemBERTa-Concat** — full end-to-end fine-tuning.
- **ChemBERTa-Concat (frozen)** — ChemBERTa weights frozen, only the linear head trains.

Note: 250k peptides × 15 epochs × 2 variants is *substantially* more compute than the lipid notebook. On a T4 expect several hours per variant. Start with `CONCAT_SEEDS = [0]`.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained('seyonec/ChemBERTa-zinc-base-v1')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

desc_scaler = MinMaxScaler().fit(train_df[DESC_COLS].values)
desc_train  = desc_scaler.transform(train_df[DESC_COLS].values).astype(np.float32)
desc_test   = desc_scaler.transform(test_df[DESC_COLS].values).astype(np.float32)

rt_scaler = MinMaxScaler()
rt_train  = rt_scaler.fit_transform(train_df[['rt']].values).astype(np.float32).ravel()
rt_test   = rt_scaler.transform(test_df[['rt']].values).astype(np.float32).ravel()

class PeptideConcatDataset(Dataset):
    def __init__(self, smiles, descriptors, rt, max_length=128):
        self.smiles = smiles
        self.descriptors = torch.tensor(descriptors, dtype=torch.float32)
        self.rt = torch.tensor(rt, dtype=torch.float32)
        self.max_length = max_length
    def __len__(self):
        return len(self.smiles)
    def __getitem__(self, idx):
        enc = tokenizer(self.smiles[idx], padding='max_length', truncation=True,
                        max_length=self.max_length, return_tensors='pt')
        return {
            'input_ids':       enc['input_ids'].squeeze(0),
            'attention_mask':  enc['attention_mask'].squeeze(0),
            'descriptors':     self.descriptors[idx],
            'rt':              self.rt[idx],
        }

train_ds = PeptideConcatDataset(train_df['smile'].tolist(), desc_train, rt_train)
test_ds  = PeptideConcatDataset(test_df['smile'].tolist(),  desc_test,  rt_test)
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=16, shuffle=False)

In [ ]:
class ChemBERTaConcatRegressor(nn.Module):
    def __init__(self, n_descriptors=7):
        super().__init__()
        self.bert = AutoModel.from_pretrained('seyonec/ChemBERTa-zinc-base-v1')
        hs = self.bert.config.hidden_size
        self.regressor = nn.Linear(hs + n_descriptors, 1)
    def forward(self, input_ids, attention_mask, descriptors):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]
        x = torch.cat([cls, descriptors], dim=-1)
        return self.regressor(x)

def evaluate(model, loader):
    model.eval()
    preds, ys = [], []
    with torch.no_grad():
        for batch in loader:
            p = model(batch['input_ids'].to(device),
                      batch['attention_mask'].to(device),
                      batch['descriptors'].to(device)).squeeze(-1).cpu().numpy()
            preds.append(p); ys.append(batch['rt'].numpy())
    p = rt_scaler.inverse_transform(np.concatenate(preds).reshape(-1, 1)).ravel()
    y = rt_scaler.inverse_transform(np.concatenate(ys).reshape(-1, 1)).ravel()
    return r2_score(y, p), mean_absolute_error(y, p)

CONCAT_SEEDS = [0]
EPOCHS = 15

concat_rows = []

for freeze_bert in [False, True]:
    label = 'ChemBERTa-Concat (frozen)' if freeze_bert else 'ChemBERTa-Concat'
    print(f'\n=== {label} ===')
    for seed in CONCAT_SEEDS:
        torch.manual_seed(seed)
        np.random.seed(seed)
        model = ChemBERTaConcatRegressor(n_descriptors=len(DESC_COLS)).to(device)
        if freeze_bert:
            for p in model.bert.parameters():
                p.requires_grad = False
        trainable = [p for p in model.parameters() if p.requires_grad]
        optimizer = torch.optim.AdamW(trainable, lr=2.5e-5)
        criterion = nn.MSELoss()

        for epoch in range(1, EPOCHS + 1):
            model.train()
            if freeze_bert:
                model.bert.eval()
            for batch in train_loader:
                optimizer.zero_grad()
                pred = model(batch['input_ids'].to(device),
                             batch['attention_mask'].to(device),
                             batch['descriptors'].to(device)).squeeze(-1)
                loss = criterion(pred, batch['rt'].to(device))
                loss.backward()
                optimizer.step()

            r2_tr, mae_tr = evaluate(model, train_loader)
            r2_te, mae_te = evaluate(model, test_loader)
            for split_name, r2, mae in [('Train', r2_tr, mae_tr), ('Test', r2_te, mae_te)]:
                concat_rows.append({'model': label, 'seed': seed, 'epoch': epoch,
                                    'split': split_name, 'metric': 'R2',  'value': r2})
                concat_rows.append({'model': label, 'seed': seed, 'epoch': epoch,
                                    'split': split_name, 'metric': 'MAE', 'value': mae})
            print(f'seed={seed}  epoch={epoch:2d}  R2_test={r2_te:.4f}  MAE_test={mae_te:.2f}')

concat_df = pd.DataFrame(concat_rows)
concat_df.to_csv('peptide_chemberta_concat_metrics.csv', index=False)

## 3. Quick comparison

Pull the existing single-task and multi-task numbers from the paper's Supplementary Data 1 (`Peptide_ChemBERTa_RDkit_Boxplot_Data_Long.csv`) and append them here for the rebuttal table.

In [ ]:
def summarise(df, group_cols=('model', 'split', 'metric')):
    g = df.groupby(list(group_cols))['value']
    return pd.DataFrame({
        'median': g.median(),
        'iqr_low':  g.quantile(0.25),
        'iqr_high': g.quantile(0.75),
        'n':        g.size(),
    }).round(4)

print('=== RDKit-only baseline (across seeds) ===')
print(summarise(baseline_df))

print('\n=== ChemBERTa-Concat (across seeds × epochs) ===')
print(summarise(concat_df))